# CalorieSense — Model Training Notebook
**AI Project (I) · National Institute of Business Management**

Phases covered:
1. Data Loading & Initial Inspection
2. Data Preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature Engineering & Selection
5. Model Training
6. Model Evaluation
7. Save Best Model

---
## Phase 1 — Data Loading & Initial Inspection

In [ ]:
# ── Cell 1: Import Libraries ─────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold

# Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Saving
import joblib

print('All libraries loaded successfully.')

In [ ]:
# ── Cell 2: Load All 4 Datasets ───────────────────────────────────────────────
# Adjust paths to wherever your data folder lives
BASE = 'data'

df_gym   = pd.read_csv(f'{BASE}/1. Gym Members Exercise Dataset/gym_members_exercise_tracking.csv')
df_fit   = pd.read_csv(f'{BASE}/2. Exercise and Fitness Metrics Dataset/exercise_dataset.csv')
df_exlib = pd.read_csv(f'{BASE}/3. Fitness Exercises Dataset/exercises_flat.csv')
df_coach = pd.read_excel(f'{BASE}/4. Mendeley Gym Recommendation Dataset/gym recommendation.xlsx')

print(f'Dataset 1 (Gym Members)          : {df_gym.shape[0]:,} rows × {df_gym.shape[1]} cols')
print(f'Dataset 2 (Exercise & Fitness)   : {df_fit.shape[0]:,} rows × {df_fit.shape[1]} cols')
print(f'Dataset 3 (Exercise Library)     : {df_exlib.shape[0]:,} rows × {df_exlib.shape[1]} cols')
print(f'Dataset 4 (Mendeley Coaching)    : {df_coach.shape[0]:,} rows × {df_coach.shape[1]} cols')
print()
print('NOTE: Datasets 3 & 4 are runtime lookup tables — NOT used for model training.')

In [ ]:
# ── Cell 3: Preview Datasets ─────────────────────────────────────────────────
print('=== Dataset 1 — Gym Members Exercise ===')
display(df_gym.head(3))

print('\n=== Dataset 2 — Exercise & Fitness Metrics ===')
display(df_fit.head(3))

In [ ]:
# ── Cell 4: Missing Value Check ───────────────────────────────────────────────
print('=== Missing Values — Dataset 1 ===')
print(df_gym.isnull().sum())

print('\n=== Missing Values — Dataset 2 ===')
print(df_fit.isnull().sum())

In [ ]:
# ── Cell 5: Descriptive Statistics ───────────────────────────────────────────
print('=== Dataset 1 .describe() ===')
display(df_gym.describe().round(2))

print('\n=== Dataset 2 .describe() ===')
display(df_fit.describe().round(2))

---
## Phase 2 — Data Preprocessing

In [ ]:
# ── Cell 6: Rename DS2 Columns to Match DS1 Naming Convention ────────────────
#
# DS1 column          DS2 equivalent        Action
# ------------------  --------------------  ---------------------------------
# Age                 Age                   same
# Gender              Gender                same
# BMI                 BMI                   same
# Weight (kg)         Actual Weight         rename
# Avg_BPM             Heart Rate            rename
# Session_Duration..  Duration (minutes)    rename + convert min→hours
# Calories_Burned     Calories Burn         rename
# —                   Dream Weight          keep (extra feature)
# —                   Exercise Intensity    keep (extra feature)
# —                   Weather Conditions    keep (extra feature)
# ID, Exercise        —                     drop (not useful for prediction)

df_fit_clean = df_fit.copy()

# Drop non-predictive columns
df_fit_clean.drop(columns=['ID', 'Exercise'], inplace=True)

# Rename to align with DS1
df_fit_clean.rename(columns={
    'Calories Burn'      : 'Calories_Burned',
    'Actual Weight'      : 'Weight (kg)',
    'Dream Weight'       : 'Dream_Weight',
    'Heart Rate'         : 'Avg_BPM',
    'Weather Conditions' : 'Weather_Conditions',
    'Exercise Intensity' : 'Exercise_Intensity',
}, inplace=True)

# Duration in DS2 is in MINUTES — convert to hours so both datasets match
df_fit_clean.rename(columns={'Duration': 'Session_Duration (hours)'}, inplace=True)
df_fit_clean['Session_Duration (hours)'] = df_fit_clean['Session_Duration (hours)'] / 60

# Tag source for traceability (optional, dropped before training)
df_gym['_source']      = 'ds1'
df_fit_clean['_source'] = 'ds2'

print('DS1 columns:', list(df_gym.columns))
print('DS2 columns (after rename):', list(df_fit_clean.columns))

In [ ]:
# ── Cell 7: Concatenate into One Training DataFrame ───────────────────────────
# pd.concat fills missing columns with NaN automatically
df = pd.concat([df_gym, df_fit_clean], ignore_index=True)

print(f'Merged DataFrame shape: {df.shape}')
print(f'  DS1 rows: {(df["_source"]=="ds1").sum()}')
print(f'  DS2 rows: {(df["_source"]=="ds2").sum()}')
print()
print('Missing values per column:')
print(df.isnull().sum().to_string())

In [ ]:
# ── Cell 8: Handle Missing Values ────────────────────────────────────────────
# Numeric columns that DS2 doesn't have → impute with median
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'Calories_Burned']

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  Filled "{col}" NaNs with median={median_val:.2f}')

# Categorical columns
for col in ['Workout_Type', 'Weather_Conditions']:
    if col in df.columns and df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f'  Filled "{col}" NaNs with mode="{mode_val}"')

# Drop source tag — not a feature
df.drop(columns=['_source'], inplace=True)

print(f'\nRemaining NaNs: {df.isnull().sum().sum()}')

In [ ]:
# ── Cell 9: Remove Duplicates & Outliers ──────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# IQR-based outlier removal on the target column
Q1 = df['Calories_Burned'].quantile(0.25)
Q3 = df['Calories_Burned'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR

before_out = len(df)
df = df[(df['Calories_Burned'] >= lower) & (df['Calories_Burned'] <= upper)]
print(f'Target outliers removed (3×IQR): {before_out - len(df)}')
print(f'Final cleaned shape: {df.shape}')

In [ ]:
# ── Cell 10: Encode Categorical Variables ─────────────────────────────────────
# Gender: Male → 1, Female → 0
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

# Workout_Type: one-hot encode (DS2 rows will be all-zero → handled by imputation already)
df = pd.get_dummies(df, columns=['Workout_Type'], prefix='wt', drop_first=False)

# Weather_Conditions: label encode (Sunny=2, Cloudy=1, Rainy=0)
weather_map = {'Sunny': 2, 'Cloudy': 1, 'Rainy': 0}
df['Weather_Conditions'] = df['Weather_Conditions'].map(weather_map)
df['Weather_Conditions'].fillna(df['Weather_Conditions'].median(), inplace=True)

# Convert bool columns (from get_dummies) to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print('Encoded columns:', [c for c in df.columns if c.startswith('wt_') or c in ['Gender','Weather_Conditions']])
print(f'DataFrame shape after encoding: {df.shape}')
display(df.head(3))

---
## Phase 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Cell 11: Target Distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Calories_Burned'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Calories Burned')
axes[0].set_xlabel('Calories Burned')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['Calories_Burned'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Calories Burned — Boxplot')
axes[1].set_ylabel('Calories Burned')

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 12: Correlation Heatmap ──────────────────────────────────────────────
numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap — All Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 13: Scatter Plots — Key Features vs Calories ─────────────────────────
scatter_features = [
    'Session_Duration (hours)', 'Avg_BPM', 'Weight (kg)', 'BMI', 'Age'
]

fig, axes = plt.subplots(1, len(scatter_features), figsize=(18, 4))
for ax, feat in zip(axes, scatter_features):
    ax.scatter(df[feat], df['Calories_Burned'],
               alpha=0.2, s=10, color='steelblue')
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('Calories Burned', fontsize=9)
    ax.set_title(feat, fontsize=9)

plt.suptitle('Feature vs Calories Burned', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 14: Feature Distribution Histograms ──────────────────────────────────
hist_features = ['Age', 'Weight (kg)', 'Height (m)', 'BMI',
                 'Session_Duration (hours)', 'Avg_BPM', 'Exercise_Intensity']
hist_features = [f for f in hist_features if f in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, feat in enumerate(hist_features):
    axes[i].hist(df[feat].dropna(), bins=30,
                 color=sns.color_palette('muted')[i % 8], edgecolor='white')
    axes[i].set_title(feat, fontsize=9)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 15: Calories by Gender & Workout Type ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gender
gender_means = df.groupby('Gender')['Calories_Burned'].mean()
axes[0].bar(['Female (0)', 'Male (1)'], gender_means.values,
            color=['#E9967A','#4682B4'])
axes[0].set_title('Average Calories Burned by Gender')
axes[0].set_ylabel('Avg Calories')

# Workout type — reconstruct label from one-hot columns
wt_cols = [c for c in df.columns if c.startswith('wt_')]
if wt_cols:
    df['_wt_label'] = df[wt_cols].idxmax(axis=1).str.replace('wt_', '')
    wt_means = df.groupby('_wt_label')['Calories_Burned'].mean().sort_values()
    wt_means.plot(kind='barh', ax=axes[1], color='steelblue')
    axes[1].set_title('Average Calories Burned by Workout Type')
    axes[1].set_xlabel('Avg Calories')
    df.drop(columns=['_wt_label'], inplace=True)

plt.tight_layout()
plt.show()

---
## Phase 4 — Feature Engineering & Selection

In [ ]:
# ── Cell 16: Feature Importance via Random Forest (quick pass) ────────────────
X_all = df.drop(columns=['Calories_Burned'])
y_all = df['Calories_Burned']

# Quick RF on full data to rank features
rf_quick = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_quick.fit(X_all, y_all)

importance_df = pd.DataFrame({
    'Feature'   : X_all.columns,
    'Importance': rf_quick.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15), x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

In [ ]:
# ── Cell 17: Select Final Features & Train/Test Split ─────────────────────────
# Keep all features — let the models decide what's useful.
# Remove low-importance or highly collinear features if needed after reviewing Cell 16.

FEATURES = [
    'Age', 'Gender', 'Weight (kg)', 'Height (m)', 'BMI',
    'Max_BPM', 'Avg_BPM', 'Resting_BPM',
    'Session_Duration (hours)',
    'Fat_Percentage', 'Water_Intake (liters)',
    'Workout_Frequency (days/week)', 'Experience_Level',
    'Exercise_Intensity', 'Weather_Conditions', 'Dream_Weight',
]
# Only keep features that actually exist (some DS1-only cols may be all-NaN)
FEATURES = [f for f in FEATURES if f in df.columns]
# Add one-hot workout type columns
wt_cols = [c for c in df.columns if c.startswith('wt_')]
FEATURES += wt_cols

X = df[FEATURES].copy()
y = df['Calories_Burned'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training set  : {X_train.shape}')
print(f'Test set      : {X_test.shape}')
print(f'Features used : {len(FEATURES)}')

In [ ]:
# ── Cell 18: Scale Features ───────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Scaling done. Train mean (first 3 cols):', X_train_sc[:, :3].mean(axis=0).round(3))

---
## Phase 5 — Model Training

In [ ]:
# ── Cell 19: Helper — Evaluate Any Model ──────────────────────────────────────
results = {}  # stores metrics for final comparison

def evaluate(name, model, X_tr, X_te, y_tr, y_te, use_scaled=False):
    Xtr = X_tr if not use_scaled else X_train_sc
    Xte = X_te if not use_scaled else X_test_sc
    
    pred = model.predict(Xte)
    mae  = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    r2   = r2_score(y_te, pred)
    
    # 5-fold cross-validation R² on full scaled set
    cv_scores = cross_val_score(model, Xtr, y_tr, cv=5,
                                scoring='r2', n_jobs=-1)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2,
                     'CV_R2_mean': cv_scores.mean(), 'CV_R2_std': cv_scores.std()}
    
    print(f'  MAE  = {mae:.2f}')
    print(f'  RMSE = {rmse:.2f}')
    print(f'  R²   = {r2:.4f}')
    print(f'  CV R² = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    return pred

print('Helper function ready.')

In [ ]:
# ── Cell 20: Model 1 — Linear Regression ─────────────────────────────────────
print('=== Linear Regression ===')
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
pred_lr = evaluate('Linear Regression', lr,
                   X_train_sc, X_test_sc, y_train, y_test,
                   use_scaled=True)

In [ ]:
# ── Cell 21: Model 2 — Random Forest Regression ───────────────────────────────
print('=== Random Forest Regression ===')
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)  # RF doesn't need scaling
pred_rf = evaluate('Random Forest', rf,
                   X_train, X_test, y_train, y_test)

In [ ]:
# ── Cell 22: Model 3 — XGBoost ────────────────────────────────────────────────
print('=== XGBoost Regression ===')
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False)
pred_xgb = evaluate('XGBoost', xgb,
                    X_train, X_test, y_train, y_test)

In [ ]:
# ── Cell 23: Model 4 — Artificial Neural Network ──────────────────────────────
print('=== Artificial Neural Network (ANN) ===')

n_features = X_train_sc.shape[1]

ann = Sequential([
    Dense(256, activation='relu', input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1)
])

ann.compile(optimizer='adam', loss='mse', metrics=['mae'])

early_stop = EarlyStopping(
    monitor='val_loss', patience=15,
    restore_best_weights=True, verbose=1
)

history = ann.fit(
    X_train_sc, y_train,
    validation_split=0.15,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

pred_ann = ann.predict(X_test_sc).flatten()
mae_ann  = mean_absolute_error(y_test, pred_ann)
rmse_ann = np.sqrt(mean_squared_error(y_test, pred_ann))
r2_ann   = r2_score(y_test, pred_ann)

results['ANN'] = {'MAE': mae_ann, 'RMSE': rmse_ann, 'R2': r2_ann,
                  'CV_R2_mean': float('nan'), 'CV_R2_std': float('nan')}

print(f'  MAE  = {mae_ann:.2f}')
print(f'  RMSE = {rmse_ann:.2f}')
print(f'  R²   = {r2_ann:.4f}')
print('  (CV not applicable for Keras models in this setup)')

# Training curve
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('ANN Training Curve')
plt.legend()
plt.tight_layout()
plt.show()

---
## Phase 6 — Model Evaluation

In [ ]:
# ── Cell 24: Comparison Table ─────────────────────────────────────────────────
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Model', 'MAE', 'RMSE', 'R²', 'CV R² (mean)', 'CV R² (std)']
results_df = results_df.sort_values('R²', ascending=False).reset_index(drop=True)
results_df[['MAE','RMSE','R²','CV R² (mean)','CV R² (std)']] = \
    results_df[['MAE','RMSE','R²','CV R² (mean)','CV R² (std)']].round(4)

display(results_df)

best_model_name = results_df.iloc[0]['Model']
print(f'\nBest model by R²: {best_model_name}')

In [ ]:
# ── Cell 25: Comparison Bar Chart ─────────────────────────────────────────────
metrics = ['MAE', 'RMSE', 'R²']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = sns.color_palette('muted', len(results_df))
for ax, metric in zip(axes, metrics):
    bars = ax.bar(results_df['Model'], results_df[metric], color=colors)
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 26: Actual vs Predicted — All Models ─────────────────────────────────
all_preds = {
    'Linear Regression': pred_lr,
    'Random Forest'    : pred_rf,
    'XGBoost'          : pred_xgb,
    'ANN'              : pred_ann,
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, pred) in zip(axes, all_preds.items()):
    ax.scatter(y_test, pred, alpha=0.3, s=10, color='steelblue')
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect fit')
    r2 = results[name]['R2']
    ax.set_title(f'{name}  (R²={r2:.4f})')
    ax.set_xlabel('Actual Calories')
    ax.set_ylabel('Predicted Calories')
    ax.legend(fontsize=8)

plt.suptitle('Actual vs Predicted — All Models', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 27: Residual Analysis — Best Model ───────────────────────────────────
best_pred = all_preds[best_model_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(best_pred, residuals, alpha=0.3, s=10, color='coral')
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].set_title(f'Residuals vs Predicted — {best_model_name}')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

---
## Phase 7 — Save Best Model & Preprocessing Pipeline

In [ ]:
# ── Cell 28: Save Best Model ──────────────────────────────────────────────────
os.makedirs('out', exist_ok=True)

# Map best model name to the actual fitted object
model_objects = {
    'Linear Regression': lr,
    'Random Forest'    : rf,
    'XGBoost'          : xgb,
}

if best_model_name == 'ANN':
    ann.save('out/best_model_ann.keras')
    print('ANN model saved to out/best_model_ann.keras')
else:
    best_model_obj = model_objects[best_model_name]
    joblib.dump(best_model_obj, 'out/best_model.pkl')
    print(f'{best_model_name} saved to out/best_model.pkl')

# Always save the scaler and feature list — required for inference
joblib.dump(scaler,   'out/scaler.pkl')
joblib.dump(FEATURES, 'out/feature_list.pkl')

# Save weather & workout_type encoding maps for the web app
import json
encoding_maps = {
    'weather_map'  : weather_map,
    'gender_map'   : {'Male': 1, 'Female': 0},
    'workout_types': [c.replace('wt_', '') for c in wt_cols],
    'feature_list' : FEATURES,
}
with open('out/encoding_maps.json', 'w') as f:
    json.dump(encoding_maps, f, indent=2)

print('\nAll artifacts saved:')
for fname in os.listdir('out'):
    print(f'  out/{fname}')

In [ ]:
# ── Cell 29: Quick Sanity Check — Load & Predict ─────────────────────────────
# Verify the saved model can be loaded and produces sensible output

if best_model_name != 'ANN':
    loaded_model   = joblib.load('out/best_model.pkl')
    loaded_scaler  = joblib.load('out/scaler.pkl')
    loaded_features = joblib.load('out/feature_list.pkl')

    sample = X_test.iloc[:5][loaded_features]
    sample_sc = loaded_scaler.transform(sample)

    # Note: tree-based models (RF, XGBoost) don't need scaling, but it's
    # harmless to pass scaled data — or use the unscaled sample for them.
    if best_model_name == 'Linear Regression':
        preds_check = loaded_model.predict(sample_sc)
    else:
        preds_check = loaded_model.predict(sample)

    print('Sanity check — 5 sample predictions:')
    for i, (actual, pred) in enumerate(zip(y_test.values[:5], preds_check)):
        print(f'  Sample {i+1}: actual={actual:.1f}  predicted={pred:.1f}  diff={abs(actual-pred):.1f}')
else:
    from tensorflow.keras.models import load_model
    loaded_ann = load_model('out/best_model_ann.keras')
    loaded_scaler = joblib.load('out/scaler.pkl')
    sample_sc = loaded_scaler.transform(X_test.iloc[:5][FEATURES])
    preds_check = loaded_ann.predict(sample_sc).flatten()
    print('ANN sanity check — 5 sample predictions:')
    for i, (actual, pred) in enumerate(zip(y_test.values[:5], preds_check)):
        print(f'  Sample {i+1}: actual={actual:.1f}  predicted={pred:.1f}  diff={abs(actual-pred):.1f}')

print('\nNotebook complete. Model is ready for web app integration.')